### Task 1: Random Forest
#### **Objective**
In this lab, you will implement missing parts of a `RandomForest` classifier. This classifier is an ensemble of decision trees that makes predictions using bootstrap sampling and random feature selection.

#### **Overview**
A `RandomForest` model is built using the following steps:
1. **Bootstrap Sampling**: Randomly selecting samples with replacement to train each tree.
2. **Random Feature Selection**: Choosing a subset of features for training each tree.
3. **Training Decision Trees**: Fitting multiple decision trees using the sampled dataset.
4. **Aggregating Predictions**: Using majority voting to determine the final prediction.

Your task is to implement the missing parts (marked as `TODO`) in the given code.

---

#### **SubTasks**

##### **SubTask 1: Implement Bootstrap Sampling**
- **Function to complete**: `_bootstrap_sample(self, X, y)`
- **Purpose**: Generate a bootstrap sample by randomly selecting data points **with replacement**.
- **Implementation details**:
  - Randomly sample `len(X)` instances from `X` and `y`, allowing repetition.
  - Return:
    - `X_sample`: The sampled feature matrix.
    - `y_sample`: The corresponding labels.
    - `indices`: The indices of the selected samples.
- **Steps**:
  1. Use `np.random.choice()` to sample indices from `X`.
  2. Select the corresponding rows from `X` and `y` using the sampled indices.
  3. Return `(X_sample, y_sample, indices)`.

**Hints:**
- Use `replace=True` in `np.random.choice()` to allow repeated selections.
- The number of selected indices should be equal to the number of rows in `X`.

---

##### **SubTask 2: Implement Random Feature Selection**
- **Function to complete**: `_select_random_features(self, X, n_features)`
- **Purpose**: Select a subset of features randomly to be used for training each tree.
- **Implementation details**:
  - Determine the number of features to select based on `self.max_features`:
    - `"sqrt"` → `sqrt(n_features)`
    - `"log2"` → `log2(n_features)`
    - `int` value → Use the given number of features.
  - Randomly select `max_features` feature indices.
  - Return:
    - `X_subset`: The feature matrix containing only the selected features.
    - `feature_indices`: The indices of the selected features.
- **Steps**:
  1. Compute the number of features to select (`max_features`).
  2. Randomly choose `max_features` feature indices using `np.random.choice()`.
  3. Extract the corresponding columns from `X` using these indices.
  4. Return `(X_subset, feature_indices)`.

**Hints:**
- Ensure `max_features` does not exceed `n_features`.
- Use `replace=False` in `np.random.choice()` to prevent selecting the same feature multiple times.

---

##### **SubTask 3: Train Individual Decision Trees**
- **Function to complete**: `fit(self, X, y)`
- **Purpose**: Train `n_estimators` decision trees using bootstrap samples and random feature selection.
- **Implementation details**:
  - **For each tree:**
    1. **Create a bootstrap sample** from `X` and `y` using `_bootstrap_sample()`.
    2. **Select a subset of features** from the bootstrap sample using `_select_random_features()`.
    3. **Train a `DecisionTreeClassifier`** using the subset of features.
    4. Store the trained tree and selected feature indices.

**Steps:**
1. Convert `X` and `y` into NumPy arrays if they are Pandas DataFrames/Series.
2. Initialize an empty list `self.trees` to store trained trees.
3. For each tree (`self.n_estimators`):
   - Call `_bootstrap_sample()` to generate a training sample.
   - Call `_select_random_features()` to select features for training.
   - Train a `DecisionTreeClassifier` on the sampled data.
   - Store the trained tree and the selected feature indices in `self.trees`.

**Hints:**
- Use `np.random.seed(self.seed)` for reproducibility.

In [ ]:
import numpy as np
from collections import Counter
from sklearn.tree import DecisionTreeClassifier


class RandomForest:
    """
    A basic implementation of a Random Forest algorithm for classification.

    This class builds an ensemble of decision trees using bootstrap sampling and random feature selection. 
    It combines predictions from individual trees using majority voting to produce a robust classification model.

    Attributes:
    ----------
    n_estimators : int
        Number of decision trees in the forest.
    max_depth : int or None
        Maximum depth of each tree. If None, the trees grow until pure leaves or until reaching `min_samples_split`.
    min_samples_split : int
        Minimum number of samples required to split an internal node in a tree.
    max_features : str or int
        Number of features to consider when looking for the best split.
        - "sqrt": Square root of the total number of features.
        - "log2": Base-2 logarithm of the total number of features.
        - int: A specific number of features to use.
    seed : int or None
        Random seed for reproducibility. If None, results will vary between runs.
    trees : list
        A list containing tuples of trained decision trees and their selected feature indices/names.
    """

    def __init__(self, n_estimators=10, max_depth=None, min_samples_split=2, max_features="sqrt", seed=None):
        """
        Initializes the Random Forest classifier.  #初始化随机森林

        Parameters:    #self表示引用的是类的属性，而不是方法的参数（起名字）
        ----------
        n_estimators : int, default=10
            Number of decision trees to build in the forest.
        max_depth : int or None, default=None
            Maximum depth of each decision tree. If None, trees grow until they are pure or cannot be split further.
        min_samples_split : int, default=2
            Minimum number of samples required to split an internal node.
        max_features : str or int, default="sqrt"                                          #每次分裂时考虑的特征数，控制模型复杂度，防止过度拟合
            Number of features to consider when splitting nodes:
            - "sqrt": Square root of the total number of features.
            - "log2": Base-2 logarithm of the total number of features.
            - int: Specific number of features.
        seed : int or None, default=None
            Random seed for reproducibility. If None, results may vary across runs.
        """
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features
        self.seed = seed
        self.trees = []

    def _bootstrap_sample(self, X, y):
        """
        Generates a bootstrap sample of the input dataset by sampling with replacement.

        Parameters:
        ----------
        X : numpy.ndarray
            Feature matrix with shape (n_samples, n_features).
        y : numpy.ndarray
            Target vector with shape (n_samples,).

        Returns:
        -------
        tuple
            A tuple (X_sample, y_sample, indices) where:
            - X_sample is the feature matrix of the bootstrap sample.  #x_sample是bootstrap采样的特征矩阵
            - y_sample is the target vector of the bootstrap sample.   #y_sample是bootstrap采样的目标向量
            - indices are the indices of the selected samples.
        """
        # TODO implement this function
        # Step 1: Use np.random.choice() to sample indices from X.
        n_samples = X.shape[0]
        indices = np.random.choice(n_samples, n_samples, replace=True)    #采样数量与样本数量相同，有放回采样

        # Step 2: Select the corresponding rows from X and y using the sampled indices.
        X_sample = X[indices]
        y_sample = y[indices]

        return X_sample, y_sample, indices

    def _select_random_features(self, X, n_features):
        """
        Selects a random subset of features to use for a decision tree.

        Parameters:
        ----------
        X : numpy.ndarray
            Feature matrix with shape (n_samples, n_features).
        n_features : int
            Total number of features in the dataset.

        Returns:
        -------
        tuple
            A tuple (X_subset, feature_indices) where:
            - X_subset is the feature matrix containing only the selected features.
            - feature_indices are the indices of the selected features.
        """
        # TODO implement this function
        # Step 1: Compute the number of features to select (max_features).
        if self.max_features == "sqrt":
            max_features = int(np.sqrt(n_features))
        elif self.max_features == "log2":
            max_features = int(np.log2(n_features))
        else:
            max_features = n_features

        # Step 2: Randomly choose max_features feature indices using np.random.choice().
        feature_indices = np.random.choice(n_features, max_features, replace=False)

        # Step 3: Extract the corresponding columns from X using these indices.
        X_subset = X[:, feature_indices]
        return X_subset, feature_indices

    def fit(self, X, y):
        """
        Trains the Random Forest model by creating an ensemble of decision trees.

        Each tree is trained on a bootstrap sample of the data and a random subset of features.

        Parameters:
        ----------
        X : numpy.ndarray
            Feature matrix with shape (n_samples, n_features).
        y : numpy.ndarray
            Target vector with shape (n_samples,).
        """
        self.feature_names = None

        self.trees = []
        n_features = X.shape[1]
        np.random.seed(self.seed)

        for _ in range(self.n_estimators):  #循环生成n_estimators个决策树
            # Create bootstrap sample
            # Step 1: Call _bootstrap_sample() to generate a training sample.
            X_train, y_train = self._bootstrap_sample(X, y) 
            
            # Step 2: Call _select_random_features() to select features for training.
            X_subset, feature_indices = self._select_random_features(X_train, n_features)

            # Step 3: Train a DecisionTreeClassifier on the sampled data.
            tree = DecisionTreeClassifier(max_depth=self.max_depth, min_samples_split=self.min_samples_split, random_state=0)
            tree.fit(X_subset, y_train)

            # Step 4: Store the trained tree and the selected feature indices in self.trees.
            selected_features = feature_indices
            self.trees.append((tree, selected_features))

    def predict(self, X): 
        predictions = np.zeros((X.shape[0], len(self.trees)))        #初始化预测结果矩阵
        for i, (tree, selected_features) in enumerate(self.trees):    #遍历每个决策树
            feature_indices = selected_features                        #获取特征索引
            predictions[:, i] = tree.predict(X[:, feature_indices])    #预测结果
        return np.apply_along_axis(lambda x: Counter(x).most_common(1)[0][0], axis=1, arr=predictions)  #投票决策

Test if the class works

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import roc_curve, auc, accuracy_score

# Step 1: Load Dataset
data = load_breast_cancer()
X = data.data
y = data.target

# Split the data into a training and a test set
X_mix, X_val, y_mix, y_val = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)
X_train, X_test, y_train, y_test = train_test_split(X_mix, y_mix, test_size=0.25, random_state=0, stratify=y_mix)
rf = RandomForest(n_estimators=5, max_depth=5, max_features="sqrt", seed=0)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
accuracy_score(y_test, y_pred)

### Task 2: kernel functions

#### **Objective**
In this lab, you will implement various kernel functions used in Support Vector Machines (SVM) and other machine learning algorithms. These kernels transform input data into higher-dimensional spaces, allowing linear models to capture complex patterns.

#### **Overview**
You will implement the following kernel functions:
1. **Linear Kernel**: Computes the dot product between two vectors.
2. **Polynomial Kernel**: Introduces non-linearity by raising the dot product to a power.
3. **Radial Basis Function (RBF) Kernel**: Uses an exponential function to measure similarity based on distance.
4. **Sigmoid Kernel**: Mimics the behavior of neural networks with a hyperbolic tangent function.

Your task is to implement these functions as described below.

---

#### **SubTasks**

##### **SubTask 1: Implement Linear Kernel**
- **Function to complete**: `linear_kernel(x1, x2)`
- **Purpose**: Computes the dot product of two vectors.
- **Formula**:
  $ k(x_1, x_2) = x_1^T x_2 $
- **Steps**:
  1. Compute the dot product using `np.dot()`.
  2. Return the computed value.

---

##### **SubTask 2: Implement Polynomial Kernel**
- **Function to complete**: `polynomial_kernel(x1, x2, degree=3, coef0=1)`
- **Purpose**: Computes a polynomial kernel, which introduces non-linearity.
- **Formula**:
  $ k(x_1, x_2) = (x_1^T x_2 + coef_0)^{degree} $
- **Steps**:
  1. Compute the dot product using `np.dot()`.
  2. Add `coef0` to the result.
  3. Raise the sum to the power of `degree`.
  4. Return the computed value.

---

##### **SubTask 3: Implement Radial Basis Function (RBF) Kernel**
- **Function to complete**: `rbf_kernel(x1, x2, gamma=0.1)`
- **Purpose**: Computes an RBF kernel, measuring similarity based on Euclidean distance.
- **Formula**:
  $ k(x_1, x_2) = \exp(-\gamma ||x_1 - x_2||^2) $
- **Steps**:
  1. Compute the squared Euclidean distance using `np.linalg.norm()`.
  2. Multiply the distance by `-gamma`.
  3. Compute the exponential using `np.exp()`.
  4. Return the computed value.

---

##### **SubTask 4: Implement Sigmoid Kernel**
- **Function to complete**: `sigmoid_kernel(x1, x2, alpha=0.1, coef0=0)`
- **Purpose**: Computes a sigmoid kernel, similar to neural network activation functions.
- **Formula**:
  $ k(x_1, x_2) = \tanh(\alpha (x_1^T x_2) + coef_0) $
- **Steps**:
  1. Compute the dot product using `np.dot()`.
  2. Multiply by `alpha` and add `coef0`.
  3. Apply the `np.tanh()` function.
  4. Return the computed value.


In [ ]:
import numpy as np

def linear_kernel(x1, x2):
    """
    Compute the linear kernel between two vectors.
    k(x1, x2) = x1.T @ x2
    """
    # TODO: implement the function
    return np.dot(x1, x2)

def polynomial_kernel(x1, x2, degree=3, coef0=1):
    """
    Compute the polynomial kernel between two vectors.
    k(x1, x2) = (x1.T @ x2 + coef0)^degree
    """
    # TODO: implement the function
    return (np.dot(x1, x2) + coef0) ** degree

def rbf_kernel(x1, x2, gamma=0.1):
    """
    Compute the Gaussian (RBF) kernel between two vectors.
    k(x1, x2) = exp(-gamma * ||x1 - x2||^2)
    """
    # TODO: implement the function
    return np.exp(-gamma * np.linalg.norm(x1 - x2) ** 2)

def sigmoid_kernel(x1, x2, alpha=0.1, coef0=0):
    """
    Compute the sigmoid kernel between two vectors.
    k(x1, x2) = tanh(alpha * (x1.T @ x2) + coef0)
    """
    # TODO: implement the function
    return np.tanh(alpha * np.dot(x1, x2) + coef0)


Let's test if all the function works

In [ ]:
# Example vectors
x1 = np.array([1, 2, 3])
x2 = np.array([4, 5, 6])

# Compute kernels
print("Linear Kernel:", linear_kernel(x1, x2))
print("Polynomial Kernel (degree=3):", polynomial_kernel(x1, x2, degree=3, coef0=1))
print("RBF Kernel (gamma=0.1):", rbf_kernel(x1, x2, gamma=0.1))
print("Sigmoid Kernel (alpha=0.1, coef0=0):", sigmoid_kernel(x1, x2, alpha=0.1, coef0=0))

### Task 3: Implementing SVM Kernel Selection and Hyperparameter Tuning
#### **Objective**
In this lab, you will implement two functions to help optimize Support Vector Machine (SVM) models. SVM is a powerful supervised learning algorithm used for classification and regression tasks. Your task is to implement missing parts of the functions to **select the best kernel** and **find the best hyperparameters** for an SVM classifier.

---

#### **Overview**
SVM models depend on various factors such as kernel functions and hyperparameters (`C`, `gamma`, etc.). Selecting the right configuration improves classification accuracy. In this lab, you will implement:

1. **Kernel Selection (`find_best_kernel`)**:
   - Train SVM models using different kernel functions.
   - Evaluate each model’s performance on a test set.
   - Return the best-performing kernel.

2. **Hyperparameter Tuning (`find_best_svm_params`)**:
   - Test multiple hyperparameter combinations.
   - Train SVM models for each configuration.
   - Return the best-performing parameter set.

Your task is to complete the missing parts (marked as `TODO`) in the given functions.

---

#### **SubTasks**

##### **SubTask 1: Implement Kernel Selection**
- **Function to complete**: `find_best_kernel(kernel_list, X, y, test_size=0.2, random_state=0)`
- **Purpose**: Determine the best SVM kernel by training models with different kernel functions and evaluating their accuracy.
- **Implementation details**:
  - **Split the dataset** into training and test sets using `train_test_split()`.
  - **Train an SVM model** with each kernel from `kernel_list`.
  - **Predict on the test set** and compute accuracy.
  - **Return the kernel** with the highest accuracy.
  
---

##### **SubTask 2: Implement Hyperparameter Tuning**
- **Function to complete**: `find_best_svm_params(param_dict, X, y, test_size=0.2, random_state=0)`
- **Purpose**: Find the best hyperparameter combination for an SVM model by testing different values and selecting the best-performing set.
- **Implementation details**:
  - **Generate all possible hyperparameter combinations** from `param_dict`.
  - **Split the dataset** into training and test sets.
  - **Train an SVM model** for each parameter combination.
  - **Predict on the test set** and compute accuracy.
  - **Return the best-performing hyperparameter set**.
  
**Hints:**
- Use `SVC(**param_set)` to train the model with a given parameter combination.
- Use `itertools.product(*param_dict.values())` to generate all parameter combinations.
- Keep track of the best-performing parameter set and its accuracy.


In [ ]:
from itertools import product
from sklearn.svm import SVC # type: ignore
from sklearn.model_selection import train_test_split # type: ignore
from sklearn.metrics import accuracy_score # type: ignore

def find_best_kernel(kernel_list, X, y, test_size=0.2, random_state=0):
    """
    Finds the best kernel for an SVM model using a single validation split.

    Parameters:
    - kernel_list (list): List of kernel names to test (e.g., ['linear', 'poly', 'rbf', 'sigmoid'])
    - X (array-like): Feature matrix
    - y (array-like): Target labels
    - test_size (float): Proportion of the dataset to use as the test set (default = 0.2)
    - random_state (int): Random seed for reproducibility (default = 0)

    Returns:
    - best_kernel (str): The kernel with the highest validation accuracy
    - best_score (float): The highest validation accuracy
    """
    # TODO: implement the function

    # 1. Use `train_test_split()` to split `X` and `y` into training (1-test_size) and testing (test_size) sets.
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state)

    best_score = 0
    best_kernel = None

    # 2. Iterate over each kernel in `kernel_list`.
    for kernel in kernel_list:
        # Train the SVM model
        # 3. Train an `SVC` model using the current kernel.
        model = SVC(kernel=kernel)
        model.fit(X_train, y_train)

        # 4. Make predictions on the test set.
        y_pred = model.predict(X_test)

        # 5. Calculate accuracy using `accuracy_score()`.
        score = accuracy_score(y_test, y_pred)

        # 6. Keep track of the kernel with the highest accuracy.
        if score > best_score:
            best_score = score
            best_kernel = kernel

    return best_kernel, best_score
    


def find_best_svm_params(param_dict, X, y, test_size=0.2, random_state=0):
    """
    Finds the best SVM parameters using a single validation split.

    Parameters:
    - param_dict (dict): Dictionary containing parameter lists to test.
                         Example: {'kernel': ['linear', 'rbf'], 'C': [0.1, 1, 10]}
    - X (array-like): Feature matrix
    - y (array-like): Target labels
    - test_size (float): Proportion of the dataset to use as the test set (default = 0.2)
    - random_state (int): Random seed for reproducibility (default = 0)

    Returns:
    - best_params (dict): Dictionary of the best parameter combination
    - best_score (float): The highest validation accuracy
    """
    # TODO: implement the function

    # 1. Split the dataset into training and test sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state)

    best_score = 0
    best_params = None

    # 2. Extract the hyperparameter keys from `param_dict`.
    param_keys = param_dict.keys()
    # 3. Use `itertools.product()` to generate all possible parameter combinations.
    for params in product(*param_dict.values()):
        
    # 4. Iterate over each parameter combination:
    #     - Train an `SVC` model with the given parameters.
        model = SVC(**dict(zip(param_keys, params)))
        model.fit(X_train, y_train)
    #     - Make predictions on the test set.
        y_pred = model.predict(X_test)
    #     - Compute accuracy.
        score = accuracy_score(y_test, y_pred)
    #     - Store the best-performing parameter combination.
        if score > best_score:
            best_score = score
            best_params = dict(zip(param_keys, params))
 
    # 5. Return the best hyperparameter set and its corresponding accuracy.

    # Generate all possible combinations of parameters

    return best_params, best_score

Let's test if find_best_kernel works

In [ ]:
from sklearn.datasets import make_classification
# Load dataset
X_circles, y_circles = make_classification(n_samples=200, noise=0.1, factor=0.5, random_state=0)

X, y = X_circles,y_circles

# Define list of kernels to test
kernels = ['linear', 'poly', 'rbf', 'sigmoid']

# Find the best kernel
best_kernel, best_score = find_best_kernel(kernels, X, y)

print(f"\nBest Kernel: {best_kernel} with Accuracy: {best_score:.4f}")

### Task 4: Implementing Naïve Bayes Classifier


#### **Objective**
In this lab, you will implement a **Naïve Bayes Classifier** that assumes conditional independence among features. This classifier will be based on count-based probabilities and log probabilities to prevent underflow.

#### **Overview**
You will implement the following functions:
1. **Fit Function (`fit`)**: Train the Naïve Bayes classifier by computing class priors and feature likelihoods.
2. **Class Probability Function (`_class_probability`)**: Compute the log probability of a given class given an input sample.
3. **Predict Function (`predict`)**: Predict class labels for new data samples using the computed probabilities.

Your task is to complete these functions as described below.

---

#### **Understanding Naïve Bayes**

##### **Bayes' Theorem**
Bayes' Theorem states:

$$P(C|X) = \frac{P(X|C) P(C)}{P(X)}$$

where:
- **Posterior probability (`P(C|X)`)**: The probability of class `C` given the feature set `X`. This is what we want to compute in classification.
- **Likelihood (`P(X|C)`)**: The probability of observing the feature set `X` given that the class is `C`.
- **Prior probability (`P(C)`)**: The probability of class `C` before observing any features.
- **Evidence (`P(X)`)**: The total probability of observing the feature set `X`, across all possible classes.

##### **Applying Bayes' Theorem in Naïve Bayes Classifier**
In the Naïve Bayes classifier, we compute the probability of each class `C` given the feature set `X` and select the class with the highest probability. Using Bayes' Theorem:


$$P(C|X) = \frac{P(X|C) P(C)}{P(X)}$$

##### **Ignoring `P(X)`**
Since `P(X)` is the same for all possible classes, it does not affect which class has the highest probability. Thus, for classification purposes, we can ignore `P(X)` and compute only:


$$P(C|X) \propto P(X|C) P(C)$$

This allows us to calculate the **relative probability** of each class without the need for `P(X)`, simplifying the computation.

##### **Using Log Probabilities**
When computing `P(X|C) * P(C)`, we multiply several probabilities together. However, probabilities are often very small, and multiplying many small numbers can lead to **numerical underflow** (values becoming too small for computers to handle accurately). To avoid this issue, we apply the **logarithm** to convert multiplication into addition:


$$\log (P(X|C) P(C)) = \log P(C) + \sum \log P(X_i | C)$$

##### **Laplace Smoothing**
To handle cases where a feature value never appears in a given class (i.e., `P(X|C) = 0`), we apply **Laplace smoothing** (also known as **additive smoothing**). Laplace smoothing prevents zero probabilities by adding a small value (typically `1`) to all counts:


$$P(X_i | C) = \frac{count(X_i, C) + 1}{total(C) + |V|}$$


where:
- `count(X_i, C)`: The number of times feature `X_i` appears in class `C`.
- `total(C)`: The total number of feature occurrences in class `C`.
- `|V|`: The total number of unique feature values (vocabulary size in text classification).

This ensures that no probability is ever zero, making the classifier more robust.

---

#### **SubTasks**

##### **SubTask 1: Implement `fit` Method**
- **Function to complete**: `fit(self, X, y)`
- **Purpose**: Train the Naïve Bayes classifier by computing class priors and feature likelihoods.
- **Implementation Details**:
  1. Identify unique class labels from `y`.
  2. Count occurrences of each class label to compute prior probabilities.
  3. Count occurrences of feature values for each class to compute likelihood probabilities.

---

##### **SubTask 2: Implement `_class_probability` Method**
- **Function to complete**: `_class_probability(self, x, c)`
- **Purpose**: Compute the log probability of class `c` given input sample `x`.
- **Implementation Details**:
  1. Compute the log prior probability of class `c`.
  2. Compute the log likelihood of the input sample `x` given class `c`, using Laplace smoothing.
  3. Sum log prior and log likelihood to get the final log probability.

---

##### **SubTask 3: Implement `predict` Method**
- **Function to complete**: `predict(self, X)`
- **Purpose**: Predict class labels for input data `X`.
- **Implementation Details**:
  1. Compute the log probability for each class using `_class_probability`.
  2. Choose the class with the highest log probability as the prediction.

In [ ]:
import numpy as np

class NaiveBayesClassifier:
    """
    A Naïve Bayes classifier that assumes conditional independence among features.
    This implementation uses categorical (count-based) probabilities and log probabilities to prevent underflow.
    """
    
    def __init__(self):
        """
        Initialize necessary data structures for the classifier.
        - self.classes: Stores the unique class labels.
        - self.class_counts: A dictionary to count occurrences of each class.
        - self.feature_counts: A dictionary to count occurrences of feature values for each class.
        - self.feature_totals: A dictionary to store total feature occurrences per class.
        """
        self.classes = None  # Stores unique class labels
        self.class_counts = {}  # Dictionary to store class counts
        self.feature_counts = {}  # Dictionary to store feature counts per class
        self.feature_totals = {}  # Dictionary to store total feature occurrences per class

    def fit(self, X, y):
        """
        Train the Naïve Bayes classifier using count-based probabilities.
        Steps:
        1. Identify unique class labels from `y`.
        2. Count occurrences of each class label to compute prior probabilities.
        3. Count occurrences of feature values for each class to compute likelihood probabilities.
        
        :param X: 2D list or numpy array, where each row represents a sample and each column represents a feature.
        :param y: 1D list or numpy array, where each element corresponds to the class label of a sample.
        """
        self.classes = np.unique(y)  # Step 1: Identify unique class labels
        
        for xi, label in zip(X, y):  # Iterate over each sample
            # Step 2: Count class occurrences
            if label not in self.class_counts:
                self.class_counts[label] = 0
            self.class_counts[label] += 1
            
            # Initialize feature count storage for each class if not present
            if label not in self.feature_counts:
                self.feature_counts[label] = {}
                self.feature_totals[label] = 0
            
            # Step 3: Count occurrences of each feature value for each class
            for i, value in enumerate(xi):
                if i not in self.feature_counts[label]:
                    self.feature_counts[label][i] = {}
                if value not in self.feature_counts[label][i]:
                    self.feature_counts[label][i][value] = 0
                self.feature_counts[label][i][value] += 1
                self.feature_totals[label] += 1
                        

    def _class_probability(self, x, c):
        """
        Compute the log probability of class `c` given input sample `x` using Naïve Bayes formula.
        Steps:
        1. Compute the log prior probability of class `c`.
        2. Compute the log likelihood of the input sample `x` given class `c`.
        3. Sum log prior and log likelihood to get the final log probability.
        
        :param x: 1D list or numpy array representing a single sample.
        :param c: Class label for which probability is being computed.
        :return: Computed log probability of `c` given `x`.
        """
        # Step 1: Compute log prior probability log(P(C))
        log_prior = np.log(self.class_counts[c] / sum(self.class_counts.values()))
        
        # Step 2: Compute log likelihood log(P(X|C)) by apply laplace smoothing
        log_likelihood = 0
        for i, value in enumerate(x):
            count = self.feature_counts[c][i].get(value, 0) + 1
            total = self.feature_totals[c] + len(self.feature_counts[c])
            log_likelihood += np.log(count / total)
               
        # Step 3: Compute final log probability log(P(C|X)) (ignoring P(X))
        return log_prior + log_likelihood

    def predict(self, X):
        """
        Predict class labels for given input `X`.
        Steps:
        1. Compute the log probability for each class using `_class_probability`.
        2. Choose the class with the highest log probability as the prediction.
        
        :param X: 2D list or numpy array where each row represents a sample.
        :return: 1D numpy array of predicted class labels.
        """
        y_pred = []
        for x in X:
            # Step 1: Compute log probability for each class
            log_probs = [self._class_probability(x, c) for c in self.classes]

            # Step 2: Choose the class with the highest probability
            y_pred.append(self.classes[np.argmax(log_probs)])

        return np.array(y_pred)

Let's test if the code works

In [ ]:
X_train = np.array([[1, 2], [1, 1], [2, 3], [5, 4], [5, 3], [6, 5]])
y_train = np.array([0, 0, 0, 1, 1, 1])

X_test = np.array([[1, 2], [5, 4]])

# Train model
nb = NaiveBayesClassifier()
nb.fit(X_train, y_train)

# Predict
predictions = nb.predict(X_test)
print("Predictions:", predictions) 
print("Expected prediction: [0 1]")


### Task 5: Nested cross validation
#### **Objective**
In this lab, you will implement missing parts of a `NestedCrossValidator` class. This class performs nested cross-validation for hyperparameter tuning and model evaluation.

#### **Overview**
Nested cross-validation consists of two levels:
1. **Outer Cross-Validation**: Splits the dataset into training and test sets to evaluate model performance.
2. **Inner Cross-Validation**: Further splits the training data to perform hyperparameter tuning.

Your task is to implement the missing parts (marked as `TODO`) in the given code. Note that the

---

#### **SubTasks**

##### **SubTask 1: Implement Manual Fold Creation**
- **Function to complete**: `_create_folds(self, X, y, n_splits, data_fraction=1.0)`
- **Purpose**: Manually create train-test folds for cross-validation. But this function behave a little bit different. Sometimes we have too many data and we want to use a little bit smaller part of the data. So we have data_fraction parameter. If we have **N** samples in total, then we actually use **int(data_fraction*N)** samples.
- **Implementation details**:
  - Ensure `data_fraction` is between 0.0 and 1.0.
  - Select a subset of indices based on `data_fraction`.
  - Shuffle the selected indices.
  - Split them into `n_splits` folds.
  - Return a list of `(train_indices, test_indices)`.

---

##### **SubTask 2: Implement Nested Cross-Validation**
- **Function to complete**: `fit(self, X, y, outer_data_fraction=1.0, inner_data_fraction=0.5)`
- **Purpose**: Perform nested cross-validation for model selection and evaluation.
- **Implementation details**:
  - Create outer folds using `_create_folds()`.
  - Iterate through outer folds:
    - Split data into outer train/test sets.
    - Initialize best parameters and best score.
    - Create inner folds from outer training set.
    - Iterate over hyperparameter combinations:
      - Train model using inner train sets.
      - Evaluate model using inner validation sets.
      - Store the best hyperparameter combination.
    - Train the final model on outer training data using the best parameters.
    - Evaluate the model on the outer test set.
  - Return list of outer fold accuracies and overall mean accuracy.



In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score
from itertools import product

class NestedCrossValidator:
    def __init__(self, model_class, param_grid, outer_splits=5, inner_splits=3, random_seed=None):
        """
        Initialize the nested cross-validator.

        :param model_class: The model class to be used (e.g., SVC, RandomForestClassifier).
        :param param_grid: Dictionary of hyperparameters and their values to search.
        :param outer_splits: Number of splits for the outer cross-validation.
        :param inner_splits: Number of splits for the inner cross-validation.
        :param random_seed: Random seed for reproducibility.
        """
        self.model_class = model_class
        self.param_grid = param_grid
        self.outer_splits = outer_splits
        self.inner_splits = inner_splits
        self.random_seed = random_seed

    def _create_folds(self, X, y, n_splits, data_fraction=1.0):
        """
        Manually create folds for cross-validation.

        :param X: Feature matrix.
        :param y: Target vector.
        :param n_splits: Number of splits for cross-validation.
        :param data_fraction: Fraction of data to use (between 0.0 and 1.0).
        :return: List of (train_indices, test_indices) for each fold.
        """
        # Validate data_fraction
        if not 0.0 < data_fraction <= 1.0:
            raise ValueError("data_fraction must be between 0.0 and 1.0 (exclusive).")

        np.random.seed(self.random_seed) # DO NOT DELETE THIS LINE
        total_indices = np.arange(len(X))
        subset_size = int(len(X) * data_fraction)
        # TODO: implement this function
        # Step 1: Select a subset of data indices based on data_fraction. (use np.random.choice function)
        selected_indices = np.random.choice(total_indices, subset_size, replace=False)
       
        # Step 2: Shuffle the selected indices. (use np.random.shuffle function)
        np.random.shuffle(selected_indices)
        
        # Step 3: Calculate the size for each fold split
        fold_size = len(selected_indices) // n_splits
      
        folds = []

        for i in range(n_splits):
            # Step 3: Split the shuffled indices into n_splits folds. 
            test_indices = selected_indices[i * fold_size:(i + 1) * fold_size]
            # train_indices = ...
            train_indices = np.setdiff1d(selected_indices, test_indices)
            folds.append((train_indices, test_indices))

        return folds

    def fit(self, X, y, outer_data_fraction=1.0, inner_data_fraction=0.5):
        """
        Perform nested cross-validation to evaluate model performance.

        :param X: Feature matrix.
        :param y: Target vector.
        :param outer_data_fraction: Fraction of data to use for outer loop.
        :param inner_data_fraction: Fraction of data to use for inner loop.
        :return: List of outer fold accuracies and mean accuracy.
        """
        # TODO: implement this function
        # Step 1: Create outer folds using _create_folds().
        outer_folds = self._create_folds(X, y, self.outer_splits, outer_data_fraction)

        # Step 2: Initialize a list to store outer fold results.
        outer_results = []

        # Step 3: Loop through each outer fold.
        for outer_idx, (train_idx, test_idx) in enumerate(outer_folds):
            # Step 4: Split data into outer train/test sets.
            X_train, X_test = X[train_idx], X[test_idx]

            # Inner loop for hyperparameter tuning
            # Step 5: Initialize best parameters and best score.
            best_params = None
            best_score = -np.inf

            # Step 6: Create inner folds from outer training data. (Don't forget inner_data_fraction here)
            inner_folds = self._create_folds(X_train, y[train_idx], self.inner_splits, inner_data_fraction)

            # Step 7: Iterate over hyperparameter combinations.
            for param_combination in product(*self.param_grid.values()):
                params = dict(zip(self.param_grid.keys(), param_combination))
                inner_scores = []

                # Step 8: Train model using inner train sets and evaluate using validation sets.
                for inner_train_idx, inner_val_idx in inner_folds:
                    # Split data into inner train/test sets
                    X_inner_train, X_val = X_train[inner_train_idx], X_train[inner_val_idx]
                    y_inner_train, y_val = y[train_idx][inner_train_idx], y[train_idx][inner_val_idx]
                    
                    model = self.model_class(**params)
                    # Train model with the current parameters
                    model.fit(X_inner_train, y_inner_train)

                    # Evaluate on validation set and update inner_scores
                    y_pred = model.predict(X_val)
                    accuracy = accuracy_score(y_val, y_pred)
                    inner_scores.append(accuracy)

                # Step 9: Select the best hyperparameter combination.
                mean_inner_score = np.mean(inner_scores)
                if mean_inner_score > best_score:
                    best_score = mean_inner_score
                    best_params = params

            # Step 10: Train the final model on outer training data using best parameters.
            final_model = self.model_class(**best_params)
            final_model.fit(X_train, y[train_idx])
            

            # Step 11: Evaluate the final model on outer test set and store the result.
            y_pred = final_model.predict(X_test)
            outer_accuracy = accuracy_score(y[test_idx], y_pred)
            outer_results.append(outer_accuracy)

            print(f"Outer Fold {outer_idx + 1} - Best Params: {best_params}, Accuracy: {outer_accuracy:.4f}")

        mean_outer_accuracy = np.mean(outer_results)
        print(f"\nOverall Accuracy from Nested Cross-Validation: {mean_outer_accuracy:.4f}")
        # Step 12: Return list of outer fold accuracies and overall mean accuracy.
        return outer_results, mean_outer_accuracy

Let's test if it works

In [ ]:
from sklearn.svm import SVC
from sklearn.datasets import load_iris

# Load dataset
data = load_iris()
X, y = data.data, data.target

# Define hyperparameter grid
param_grid = {
    'C': [0.1, 1, 10],
    'kernel': ['linear', 'rbf']
}

# Initialize and run nested cross-validation
nested_cv = NestedCrossValidator(model_class=SVC, param_grid=param_grid, outer_splits=5, inner_splits=3, random_seed=0)
outer_results, meanaccuracy = nested_cv.fit(X, y)

The output in my laptop is:
```
Outer Fold 1 - Best Params: {'C': 10, 'kernel': 'rbf'}, Accuracy: 0.9667
Outer Fold 2 - Best Params: {'C': 1, 'kernel': 'linear'}, Accuracy: 1.0000
Outer Fold 3 - Best Params: {'C': 10, 'kernel': 'rbf'}, Accuracy: 0.9667
Outer Fold 4 - Best Params: {'C': 0.1, 'kernel': 'linear'}, Accuracy: 0.9333
Outer Fold 5 - Best Params: {'C': 0.1, 'kernel': 'linear'}, Accuracy: 1.0000

Overall Accuracy from Nested Cross-Validation: 0.9733
```